<a href="https://colab.research.google.com/github/iescobedo5/Computer-Vision/blob/main/new_faster_5k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q /content/drive/MyDrive/subset5k.zip -d /content/subset5k

In [ ]:
!ls /content/subset5k/content/UADETRAC/DETRAC_Upload/data/subsets/subset5k

dataset.yaml  images  labels


In [ ]:
!pip install -q torchmetrics pycocotools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 24.3 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import time

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
DATA_ROOT = Path("/content/subset5k/content/UADETRAC/DETRAC_Upload/data/subsets/subset5k")

TRAIN_IMG_DIR = DATA_ROOT / "images" / "train"
TRAIN_LBL_DIR = DATA_ROOT / "labels" / "train"
VAL_IMG_DIR   = DATA_ROOT / "images" / "val"
VAL_LBL_DIR   = DATA_ROOT / "labels" / "val"

SAVE_DIR = Path("/content/drive/MyDrive/UA/fasterrcnn_results/subset5k")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("Train images:", TRAIN_IMG_DIR)
print("Train labels:", TRAIN_LBL_DIR)
print("Val images:", VAL_IMG_DIR)
print("Val labels:", VAL_LBL_DIR)

Train images: /content/subset5k/content/UADETRAC/DETRAC_Upload/data/subsets/subset5k/images/train
Train labels: /content/subset5k/content/UADETRAC/DETRAC_Upload/data/subsets/subset5k/labels/train
Val images: /content/subset5k/content/UADETRAC/DETRAC_Upload/data/subsets/subset5k/images/val
Val labels: /content/subset5k/content/UADETRAC/DETRAC_Upload/data/subsets/subset5k/labels/val


In [ ]:
class YOLODetectionDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.image_files = sorted([
            p for p in self.img_dir.iterdir()
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        label_path = self.lbl_dir / f"{img_path.stem}.txt"

        image = Image.open(img_path).convert("RGB")
        w, h = image.size

        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue

                    cls, x_center, y_center, bw, bh = map(float, parts)

                    x_center *= w
                    y_center *= h
                    bw *= w
                    bh *= h

                    x1 = x_center - bw / 2
                    y1 = y_center - bh / 2
                    x2 = x_center + bw / 2
                    y2 = y_center + bh / 2

                    x1 = max(0, min(x1, w))
                    y1 = max(0, min(y1, h))
                    x2 = max(0, min(x2, w))
                    y2 = max(0, min(y2, h))

                    if x2 <= x1 or y2 <= y1:
                        continue

                    boxes.append([x1, y1, x2, y2])

                    labels.append(int(cls) + 1)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) if len(boxes) > 0 else torch.zeros((0,), dtype=torch.float32),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64)
        }

        image = torchvision.transforms.functional.to_tensor(image)
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
train_dataset = YOLODetectionDataset(TRAIN_IMG_DIR, TRAIN_LBL_DIR)
val_dataset = YOLODetectionDataset(VAL_IMG_DIR, VAL_LBL_DIR)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))

Train dataset size: 5000
Val dataset size: 1000


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

In [ ]:
num_classes = 2  # background + vehicle

model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

model.to(device)
print("Model ready.")

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 148MB/s]


Model ready.


In [ ]:
params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

In [ ]:
def train_one_epoch(model, loader, optimizer, device, epoch):
    model.train()
    running_loss = 0.0

    for step, (images, targets) in enumerate(loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running_loss += losses.item()

        if (step + 1) % 100 == 0:
            print(f"Epoch [{epoch}] Step [{step+1}/{len(loader)}] Loss: {losses.item():.4f}")

    return running_loss / len(loader)


@torch.no_grad()
def validate_one_epoch(model, loader, device, epoch):
    model.train()
    running_loss = 0.0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        running_loss += losses.item()

    avg_loss = running_loss / len(loader)
    print(f"Epoch [{epoch}] Validation Loss: {avg_loss:.4f}")
    return avg_loss


@torch.no_grad()
def evaluate_map(model, loader, device, score_thresh=0.0):
    model.eval()
    metric = MeanAveragePrecision()

    for images, targets in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        preds = []
        gts = []

        for output, target in zip(outputs, targets):
            boxes = output["boxes"].detach().cpu()
            scores = output["scores"].detach().cpu()
            labels = output["labels"].detach().cpu()

            if score_thresh > 0:
                keep = scores >= score_thresh
                boxes = boxes[keep]
                scores = scores[keep]
                labels = labels[keep]

            preds.append({
                "boxes": boxes,
                "scores": scores,
                "labels": labels,
            })

            gts.append({
                "boxes": target["boxes"].detach().cpu(),
                "labels": target["labels"].detach().cpu(),
            })

        metric.update(preds, gts)

    return metric.compute()


@torch.no_grad()
def evaluate_precision_recall(model, loader, device, score_thresh=0.5, iou_thresh=0.5):
    model.eval()

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for output, target in zip(outputs, targets):
            pred_boxes = output["boxes"].detach().cpu()
            pred_scores = output["scores"].detach().cpu()
            pred_labels = output["labels"].detach().cpu()

            gt_boxes = target["boxes"].detach().cpu()
            gt_labels = target["labels"].detach().cpu()

            keep = pred_scores >= score_thresh
            pred_boxes = pred_boxes[keep]
            pred_labels = pred_labels[keep]

            matched_gt = set()

            for pb, pl in zip(pred_boxes, pred_labels):
                same_class_idx = (gt_labels == pl).nonzero(as_tuple=True)[0]

                best_iou = 0.0
                best_gt_idx = -1

                for gt_idx in same_class_idx:
                    gt_idx = gt_idx.item()
                    if gt_idx in matched_gt:
                        continue

                    iou = box_iou(pb.unsqueeze(0), gt_boxes[gt_idx].unsqueeze(0))[0, 0].item()

                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = gt_idx

                if best_iou >= iou_thresh:
                    total_tp += 1
                    matched_gt.add(best_gt_idx)
                else:
                    total_fp += 1

            total_fn += len(gt_boxes) - len(matched_gt)

    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall = total_tp / (total_tp + total_fn + 1e-8)

    return precision, recall


@torch.no_grad()
def predict_sample(model, dataset, device, idx=0, score_thresh=0.5):
    model.eval()
    img, target = dataset[idx]
    pred = model([img.to(device)])[0]

    print("Ground truth boxes:", len(target["boxes"]))
    print("Predicted boxes above threshold:", (pred["scores"].cpu() >= score_thresh).sum().item())

    return img, target, pred

In [ ]:
num_epochs = 5
best_val_loss = float("inf")

train_losses = []
val_losses = []

for epoch in range(1, num_epochs + 1):
    start_time = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch)
    val_loss = validate_one_epoch(model, val_loader, device, epoch)

    lr_scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch}/{num_epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    torch.save(model.state_dict(), SAVE_DIR / "fasterrcnn_last_5k_real.pth")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_DIR / "fasterrcnn_best_5k.pth")
        print("Saved best model.")

    elapsed = time.time() - start_time
    print(f"Epoch time: {elapsed/60:.2f} min\n")

Epoch [1] Step [100/1250] Loss: 0.3630
Epoch [1] Step [200/1250] Loss: 0.2490
Epoch [1] Step [300/1250] Loss: 0.4688
Epoch [1] Step [400/1250] Loss: 0.3569
Epoch [1] Step [500/1250] Loss: 0.1918
Epoch [1] Step [600/1250] Loss: 0.2350
Epoch [1] Step [700/1250] Loss: 0.2669
Epoch [1] Step [800/1250] Loss: 0.3091
Epoch [1] Step [900/1250] Loss: 0.2999
Epoch [1] Step [1000/1250] Loss: 0.3436
Epoch [1] Step [1100/1250] Loss: 0.2231
Epoch [1] Step [1200/1250] Loss: 0.2543
Epoch [1] Validation Loss: 0.2519
Epoch [1/5] Train Loss: 0.2974 | Val Loss: 0.2519
Saved best model.
Epoch time: 27.79 min

Epoch [2] Step [100/1250] Loss: 0.2625
Epoch [2] Step [200/1250] Loss: 0.3417
Epoch [2] Step [300/1250] Loss: 0.2563
Epoch [2] Step [400/1250] Loss: 0.2072
Epoch [2] Step [500/1250] Loss: 0.2518
Epoch [2] Step [600/1250] Loss: 0.3247
Epoch [2] Step [700/1250] Loss: 0.1904
Epoch [2] Step [800/1250] Loss: 0.2306
Epoch [2] Step [900/1250] Loss: 0.1448
Epoch [2] Step [1000/1250] Loss: 0.1621
Epoch [2] Ste

In [ ]:
print("Loading best model for final evaluation...")
model.load_state_dict(torch.load(SAVE_DIR / "fasterrcnn_best_5k.pth", map_location=device))
model.to(device)

map_results = evaluate_map(model, val_loader, device)
precision, recall = evaluate_precision_recall(
    model,
    val_loader,
    device,
    score_thresh=0.5,
    iou_thresh=0.5
)

print("\nFinal Validation Results:")
print(f"mAP@0.50:      {map_results['map_50']:.4f}")
print(f"mAP@0.50:0.95: {map_results['map']:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")

Loading best model for final evaluation...

Final Validation Results:
mAP@0.50:      0.9857
mAP@0.50:0.95: 0.8053
Precision:     0.9255
Recall:        0.9849
